In [ ]:
import json
import cv2
import sys
import torch
sys.path.append("/Users/abhinavarora/Desktop/CadenceCV/src")
import yolo_model_loader
with open("/Users/abhinavarora/Desktop/CadenceCV/data/strikefoot_data.json", "r") as file:
    data = json.load(file)

model = yolo_model_loader.load_model("/Users/abhinavarora/Desktop/CadenceCV/models/21_keypoints.pt")

base_video_dir = "/Users/abhinavarora/Desktop/CadenceCV/Videos"
video_extensions = [".mp4", ".MOV"]
for strike in data:
    for ext in video_extensions:
        try:
            dir = base_video_dir + "/" + strike["video"] + ext
            cap = cv2.VideoCapture(dir)
            cap.set(cv2.CAP_PROP_POS_FRAMES, strike["frame"])
            ret, img = cap.read()
            source = model.predict(img, conf=0.3)

            for result in source:
                kpts = result.keypoints.xy[0]
                conf = result.keypoints.conf[0]

            if strike["side"] == "R":
                if conf[18] > 0.25 and conf[20] > 0.25:
                    foot_vector = kpts[18] - kpts[20]
                    angle = torch.atan2(-foot_vector[1], foot_vector[0])
                    strike["foot_angle"] = torch.rad2deg(angle).item()
                else:
                    strike["foot_angle"] = None
            else:
                if conf[17] > 0.25 and conf[19] > 0.25:
                    foot_vector = kpts[17] - kpts[19]
                    angle = torch.atan2(-foot_vector[1], foot_vector[0])
                    strike["foot_angle"] = torch.rad2deg(angle).item()
                else:
                    strike["foot_angle"] = None
            break

        except Exception:
            continue
            

In [10]:
for strike in data:
    del strike["left_foot_angle"]
    del strike["right_foot_angle"]

In [11]:
with open("/Users/abhinavarora/Desktop/CadenceCV/data/strikefoot_data.json", "w") as file:
    json.dump(data, file, indent=4)